# Hand Detail Experiment

Ce notebook reprend le script `hand_detail_experiment.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Teste si les landmarks de main ameliorent la prediction causale d'entree.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Hand-detail repeated-split experiment using MediaPipe Hand Landmarker.
- Run par defaut : `runs/exp_103_hand_detail_repeated_split`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "hand_detail_experiment.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import math
import time
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import torch
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from ml_pipeline import ROOT, load_json, write_json, zone_polygon
from sequence_experiments import set_seed
from time_to_entry_experiments import (
    add_selection_score,
    ap_auc_rows,
    create_run_dir,
    device_from_arg,
    evaluate_causal,
    make_multibin_targets,
    make_survival_targets,
    multibin_score_frame,
    score_columns_for_objective,
    survival_score_frame,
    train_one,
)
from time_to_entry_repeated_split_experiments import (
    repeated_split_maps,
    summarize,
    with_base_model_name,
    write_summary as write_repeated_summary,
)


HAND_MODEL = ROOT / "hand_landmarker.task"
HAND_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
TIP_SPECS = {
    "thumb_tip": 4,
    "index_tip": 8,
    "middle_tip": 12,
    "ring_tip": 16,
    "pinky_tip": 20,
}
SIDE_NAMES = ["left", "right"]


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `ensure_hand_model`

Cette cellule definit `ensure_hand_model`. Elle prepare une partie du script.

In [ ]:
def ensure_hand_model():
    if not HAND_MODEL.exists():
        urllib.request.urlretrieve(HAND_MODEL_URL, HAND_MODEL)
    return HAND_MODEL


## Fonction `hand_feature_columns`

Cette cellule definit `hand_feature_columns`. Elle prepare une partie du script.

In [ ]:
def hand_feature_columns():
    cols = []
    for side in SIDE_NAMES:
        prefix = f"hand_{side}"
        cols.extend(
            [
                f"{prefix}_present",
                f"{prefix}_score",
                f"{prefix}_wrist_x_norm",
                f"{prefix}_wrist_y_norm",
                f"{prefix}_wrist_z",
                f"{prefix}_wrist_signed_dist_norm",
                f"{prefix}_wrist_inside",
            ]
        )
        for tip_name in TIP_SPECS:
            cols.extend(
                [
                    f"{prefix}_{tip_name}_x_norm",
                    f"{prefix}_{tip_name}_y_norm",
                    f"{prefix}_{tip_name}_z",
                    f"{prefix}_{tip_name}_signed_dist_norm",
                    f"{prefix}_{tip_name}_inside",
                ]
            )
        cols.extend(
            [
                f"{prefix}_min_tip_signed_dist_norm",
                f"{prefix}_any_tip_inside",
                f"{prefix}_mean_tip_z",
                f"{prefix}_min_tip_z",
                f"{prefix}_wrist_to_middle_tip_norm",
            ]
        )
    return cols


## Fonction `empty_hand_row`

Cette cellule definit `empty_hand_row`. Elle prepare une partie du script.

In [ ]:
def empty_hand_row(video_id, split, path, frame_idx, time_s):
    row = {
        "video_id": video_id,
        "split": split,
        "path": path,
        "frame": int(frame_idx),
        "time_s": float(time_s),
    }
    for col in hand_feature_columns():
        row[col] = 0.0
    return row


## Fonction `signed_distance_norm`

Cette cellule definit `signed_distance_norm`. Elle prepare une partie du script.

In [ ]:
def signed_distance_norm(x, y, polygon, diag):
    if np.isnan(x) or np.isnan(y):
        return np.nan
    signed = float(cv2.pointPolygonTest(polygon, (float(x), float(y)), True))
    return signed / diag


## Fonction `landmark_xy_norm`

Cette cellule definit `landmark_xy_norm`. Elle prepare une partie du script.

In [ ]:
def landmark_xy_norm(landmark):
    return float(landmark.x), float(landmark.y), float(landmark.z)


## Fonction `assign_hands`

Cette cellule definit `assign_hands`. Elle prepare une partie du script.

In [ ]:
def assign_hands(result, pose_row, width, height):
    if not result.hand_landmarks:
        return {}
    wrist_targets = {
        "left": np.array([float(pose_row.get("left_wrist_x", np.nan)), float(pose_row.get("left_wrist_y", np.nan))], dtype=np.float32),
        "right": np.array([float(pose_row.get("right_wrist_x", np.nan)), float(pose_row.get("right_wrist_y", np.nan))], dtype=np.float32),
    }
    assigned = {}
    used = set()
    for det_idx, landmarks in enumerate(result.hand_landmarks):
        wrist = landmarks[0]
        wrist_px = np.array([float(wrist.x * width), float(wrist.y * height)], dtype=np.float32)
        distances = []
        for side in SIDE_NAMES:
            target = wrist_targets[side]
            if np.isnan(target).any() or side in used:
                continue
            distances.append((float(np.linalg.norm(wrist_px - target)), side))
        if distances:
            _, side = min(distances, key=lambda x: x[0])
        else:
            handed = ""
            if result.handedness and det_idx < len(result.handedness) and result.handedness[det_idx]:
                handed = str(result.handedness[det_idx][0].category_name).lower()
            if handed == "left" and "left" not in used:
                side = "left"
            elif handed == "right" and "right" not in used:
                side = "right"
            else:
                side = "left" if "left" not in used else "right"
        assigned[side] = det_idx
        used.add(side)
    return assigned


## Fonction `hand_rows_for_video`

Cette cellule definit `hand_rows_for_video`. Elle prepare une partie du script.

In [ ]:
def hand_rows_for_video(video_df, hand_model_path, polygon):
    options = vision.HandLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=str(hand_model_path)),
        running_mode=vision.RunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.35,
        min_hand_presence_confidence=0.35,
        min_tracking_confidence=0.35,
    )
    rows = []
    video_path = resolve(str(video_df["path"].iloc[0]))
    split = str(video_df["split"].iloc[0])
    video_id = str(video_df["video_id"].iloc[0])
    fps = float(video_df["fps"].iloc[0])
    width = float(video_df["width"].iloc[0])
    height = float(video_df["height"].iloc[0])
    diag = math.sqrt(width * width + height * height)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    ts = 0
    with vision.HandLandmarker.create_from_options(options) as landmarker:
        target_rows = video_df.sort_values("frame").reset_index(drop=True)
        target_frames = {int(frame): idx for idx, frame in enumerate(target_rows["frame"].tolist())}
        max_target_frame = int(target_rows["frame"].max()) if len(target_rows) else -1
        frame_idx = -1
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame_idx += 1
            if frame_idx > max_target_frame:
                break
            if frame_idx not in target_frames:
                continue
            pose_row = target_rows.iloc[target_frames[frame_idx]]
            ts += max(1, int(round(1000.0 / max(fps, 1.0))))
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(image, ts)
            row = empty_hand_row(video_id, split, str(video_df["path"].iloc[0]), int(pose_row["frame"]), float(pose_row["time_s"]))
            assignments = assign_hands(result, pose_row, width, height)
            for side, det_idx in assignments.items():
                prefix = f"hand_{side}"
                hand_score = 0.0
                if result.handedness and det_idx < len(result.handedness) and result.handedness[det_idx]:
                    hand_score = float(result.handedness[det_idx][0].score)
                landmarks = result.hand_landmarks[det_idx]
                wrist = landmarks[0]
                wx, wy, wz = landmark_xy_norm(wrist)
                row[f"{prefix}_present"] = 1.0
                row[f"{prefix}_score"] = hand_score
                row[f"{prefix}_wrist_x_norm"] = wx
                row[f"{prefix}_wrist_y_norm"] = wy
                row[f"{prefix}_wrist_z"] = wz
                wrist_signed = signed_distance_norm(wx * width, wy * height, polygon, diag)
                row[f"{prefix}_wrist_signed_dist_norm"] = 0.0 if np.isnan(wrist_signed) else float(wrist_signed)
                row[f"{prefix}_wrist_inside"] = 1.0 if (not np.isnan(wrist_signed) and wrist_signed >= 0) else 0.0
                tip_signed = []
                tip_z = []
                middle_tip_xy = None
                for tip_name, tip_idx in TIP_SPECS.items():
                    tx, ty, tz = landmark_xy_norm(landmarks[tip_idx])
                    row[f"{prefix}_{tip_name}_x_norm"] = tx
                    row[f"{prefix}_{tip_name}_y_norm"] = ty
                    row[f"{prefix}_{tip_name}_z"] = tz
                    signed = signed_distance_norm(tx * width, ty * height, polygon, diag)
                    if np.isnan(signed):
                        signed = -1.0
                    row[f"{prefix}_{tip_name}_signed_dist_norm"] = float(signed)
                    row[f"{prefix}_{tip_name}_inside"] = 1.0 if signed >= 0 else 0.0
                    tip_signed.append(float(signed))
                    tip_z.append(float(tz))
                    if tip_name == "middle_tip":
                        middle_tip_xy = (tx, ty)
                row[f"{prefix}_min_tip_signed_dist_norm"] = float(min(tip_signed)) if tip_signed else -1.0
                row[f"{prefix}_any_tip_inside"] = 1.0 if any(value >= 0 for value in tip_signed) else 0.0
                row[f"{prefix}_mean_tip_z"] = float(np.mean(tip_z)) if tip_z else 0.0
                row[f"{prefix}_min_tip_z"] = float(np.min(tip_z)) if tip_z else 0.0
                if middle_tip_xy is not None:
                    row[f"{prefix}_wrist_to_middle_tip_norm"] = float(np.linalg.norm(np.array([wx, wy]) - np.array(middle_tip_xy)))
            rows.append(row)
    cap.release()
    return rows


## Fonction `extract_hand_features`

Cette cellule definit `extract_hand_features`. Elle prepare une partie du script.

In [ ]:
def extract_hand_features(run_dir, pose_csv, zones_json):
    pose = pd.read_csv(pose_csv)
    zones = load_json(zones_json)
    polygon = zone_polygon(zones)
    hand_model_path = ensure_hand_model()
    all_rows = []
    timings = []
    for video_id, group in pose.groupby("video_id", sort=False):
        t0 = time.perf_counter()
        rows = hand_rows_for_video(group, hand_model_path, polygon)
        timings.append(
            {
                "video_id": video_id,
                "frames": int(len(rows)),
                "seconds": float(time.perf_counter() - t0),
            }
        )
        all_rows.extend(rows)
        print(f"[hand] {video_id} frames={len(rows)}")
    hand_df = pd.DataFrame(all_rows)
    hand_path = run_dir / "features" / "hand_frame_features.csv"
    hand_df.to_csv(hand_path, index=False)
    timing_df = pd.DataFrame(timings)
    timing_df["fps"] = timing_df["frames"] / timing_df["seconds"].clip(lower=1e-6)
    timing_df.to_csv(run_dir / "metrics" / "hand_extraction_timing.csv", index=False)
    return hand_df, timing_df


## Fonction `add_hand_motion_features`

Cette cellule definit `add_hand_motion_features`. Elle prepare une partie du script.

In [ ]:
def add_hand_motion_features(frame_df):
    rows = []
    base_cols = [
        col
        for col in frame_df.columns
        if col.startswith("hand_")
        and (
            col.endswith("_x_norm")
            or col.endswith("_y_norm")
            or col.endswith("_z")
            or col.endswith("_signed_dist_norm")
            or col.endswith("_wrist_to_middle_tip_norm")
        )
    ]
    for _, group in frame_df.groupby("video_id", sort=False):
        group = group.sort_values("frame").copy()
        fps = float(group["fps"].iloc[0])
        for col in base_cols:
            group[f"{col}_vel"] = group[col].diff().fillna(0.0) * fps
        rows.append(group)
    return pd.concat(rows, ignore_index=True)


## Fonction `combined_feature_columns`

Cette cellule definit `combined_feature_columns`. Elle prepare une partie du script.

In [ ]:
def combined_feature_columns(frame_df):
    cols = [
        "person_conf",
        "bbox_area_norm",
        "max_signed_dist_norm",
        "max_signed_dist_vel",
        "max_signed_dist_acc",
    ]
    for prefix in [
        "left_wrist",
        "right_wrist",
        "left_elbow",
        "right_elbow",
        "head",
        "torso",
    ]:
        cols.extend(
            [
                f"{prefix}_x_norm",
                f"{prefix}_y_norm",
                f"{prefix}_conf",
                f"{prefix}_signed_dist_norm",
                f"{prefix}_inside",
                f"{prefix}_x_vel",
                f"{prefix}_y_vel",
                f"{prefix}_signed_dist_vel",
            ]
        )
    for col in hand_feature_columns():
        cols.append(col)
        if col.endswith("_x_norm") or col.endswith("_y_norm") or col.endswith("_z") or col.endswith("_signed_dist_norm") or col.endswith("_wrist_to_middle_tip_norm"):
            vel_col = f"{col}_vel"
            if vel_col in frame_df.columns:
                cols.append(vel_col)
    return [col for col in cols if col in frame_df.columns]


## Fonction `build_sequence_X`

Cette cellule definit `build_sequence_X`. Elle prepare une partie du script.

In [ ]:
def build_sequence_X(frame_df, sequence_index_path, seq_len=10):
    meta = pd.read_csv(sequence_index_path)
    frame_df = frame_df.copy()
    frame_df = frame_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    feature_cols = combined_feature_columns(frame_df)
    by_video = {}
    for video_id, group in frame_df.groupby("video_id", sort=False):
        group = group.sort_values("frame").reset_index(drop=True)
        frame_to_pos = {int(frame): idx for idx, frame in enumerate(group["frame"].tolist())}
        by_video[video_id] = (group, frame_to_pos)
    X = np.zeros((len(meta), seq_len, len(feature_cols)), dtype=np.float32)
    keep_mask = np.zeros(len(meta), dtype=bool)
    for idx, row in meta.iterrows():
        video_id = row["video_id"]
        frame = int(row["frame"])
        if video_id not in by_video:
            continue
        group, frame_to_pos = by_video[video_id]
        if frame not in frame_to_pos:
            continue
        end_pos = frame_to_pos[frame]
        start_pos = max(0, end_pos - seq_len + 1)
        seq = group.iloc[start_pos : end_pos + 1][feature_cols].to_numpy(dtype=np.float32)
        if len(seq) < seq_len:
            pad = np.repeat(seq[:1], seq_len - len(seq), axis=0) if len(seq) else np.zeros((seq_len, len(feature_cols)), dtype=np.float32)
            seq = np.vstack([pad, seq])
        X[idx] = seq[-seq_len:]
        keep_mask[idx] = True
    meta = meta[keep_mask].reset_index(drop=True)
    X = X[keep_mask]
    return X, meta, feature_cols


## Fonction `enrich_sequence_meta`

Cette cellule definit `enrich_sequence_meta`. Elle prepare une partie du script.

In [ ]:
def enrich_sequence_meta(meta, entry_times_csv):
    meta = meta.copy()
    entry_times_csv = resolve(entry_times_csv)
    if entry_times_csv.exists():
        entry = pd.read_csv(entry_times_csv)
        keep = [
            "video_id",
            "event_type",
            "target_source",
            "body_part",
            "spatial_relation",
            "physical_entry_time_s",
            "risk_onset_time_s",
        ]
        keep = [col for col in keep if col in entry.columns]
        meta = meta.merge(entry[keep], on="video_id", how="left")
    else:
        meta["event_type"] = ""
    meta["target_time_numeric"] = pd.to_numeric(meta["target_time_s"], errors="coerce")
    meta["time_to_target_numeric"] = pd.to_numeric(meta["time_to_target_s"], errors="coerce")
    meta["has_physical_entry"] = meta["is_danger_clip"].astype(int).eq(1) & meta["target_time_numeric"].notna()
    meta["is_hard_negative"] = meta.get("event_type", pd.Series("", index=meta.index)).fillna("").eq("near_miss")
    return meta


## Fonction `sampled_pose_rows`

Cette cellule definit `sampled_pose_rows`. Elle prepare une partie du script.

In [ ]:
def sampled_pose_rows(pose_df, frame_stride, frame_offset, allowed_video_ids):
    pose_df = pose_df[pose_df["video_id"].isin(allowed_video_ids)].copy()
    return pose_df[pose_df["frame"].astype(int).mod(frame_stride).eq(frame_offset)].copy()


## Fonction `evaluate_prediction_frame`

Cette cellule definit `evaluate_prediction_frame`. Elle prepare une partie du script.

In [ ]:
def evaluate_prediction_frame(pred, score_cols, thresholds, persistence_values, repeat_seed):
    rows = []
    for score_col in score_cols:
        for threshold in thresholds:
            for persistence in persistence_values:
                for split in ["val", "test"]:
                    row = evaluate_causal(pred, score_col, threshold, split, persistence)
                    row["objective"] = str(pred["objective"].iloc[0])
                    row["model_name"] = str(pred["model_name"].iloc[0])
                    row["inference_ms_per_window"] = float(pred["inference_ms_per_window"].iloc[0])
                    row["train_time_s"] = float(pred["train_time_s"].iloc[0])
                    row["repeat_seed"] = int(repeat_seed)
                    rows.append(row)
    return rows


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    set_seed(args.seed)
    device = device_from_arg(args.device)
    run_dir = create_run_dir(args.run_name)
    pose_csv = resolve(args.pose_csv)
    zones_json = resolve(args.zones_json)
    sequence_index_path = resolve(args.sequence_index)
    split_maps = repeated_split_maps(args.old_score_run)
    thresholds = [round(float(x), 2) for x in np.arange(args.threshold_min, args.threshold_max + 1e-9, args.threshold_step)]
    persistence_values = [int(x) for x in args.persistence_windows]

    write_json(
        run_dir / "config.json",
        {
            "pose_csv": str(pose_csv),
            "zones_json": str(zones_json),
            "sequence_index": str(sequence_index_path),
            "old_score_run": str(resolve(args.old_score_run)),
            "device": str(device),
            "model_specs": args.model_specs,
        },
    )

    base_index = pd.read_csv(sequence_index_path)
    frame_offset = int(base_index["frame"].astype(int).min() % args.frame_stride)
    timing_df = pd.DataFrame()
    if args.frame_features_csv:
        merged_path = resolve(args.frame_features_csv)
        merged = pd.read_csv(merged_path)
        hand_cols = [col for col in merged.columns if col.startswith("hand_")]
    else:
        pose = pd.read_csv(pose_csv)
        pose = sampled_pose_rows(pose, args.frame_stride, frame_offset, set(base_index["video_id"].unique().tolist()))
        sampled_pose_path = run_dir / "features" / "pose_features_sampled.csv"
        pose.to_csv(sampled_pose_path, index=False)
        hand_df, timing_df = extract_hand_features(run_dir, sampled_pose_path, zones_json)
        merged = pose.merge(
            hand_df.drop(columns=["split", "path", "time_s"]),
            on=["video_id", "frame"],
            how="left",
        )
        hand_cols = [col for col in hand_df.columns if col.startswith("hand_")]
        for col in hand_cols:
            if col not in merged:
                merged[col] = 0.0
        merged[hand_cols] = merged[hand_cols].fillna(0.0)
        merged = add_hand_motion_features(merged)
        merged_path = run_dir / "features" / "pose_hand_frame_features.csv"
        merged.to_csv(merged_path, index=False)

    X_raw, base_meta, feature_cols = build_sequence_X(merged, sequence_index_path, seq_len=args.seq_len)
    base_meta = enrich_sequence_meta(base_meta, args.entry_times_csv)
    multibin_y, multibin_weights = make_multibin_targets(base_meta)
    survival_y, survival_weights = make_survival_targets(base_meta)

    all_metric_rows = []
    all_ap_rows = []
    all_history = []
    train_rows = []
    split_audit_rows = []
    for repeat_seed, split_map in sorted(split_maps.items()):
        meta = base_meta.copy()
        meta["split"] = meta["video_id"].map(split_map)
        train_mask = meta["split"].to_numpy() == "train"
        flat = X_raw[train_mask].reshape(-1, X_raw.shape[-1])
        mean = flat.mean(axis=0)
        std = flat.std(axis=0)
        std = np.where(std > 1e-6, std, 1.0)
        X = ((X_raw - mean) / std).astype(np.float32)
        meta.to_csv(run_dir / "features" / f"split_seed_{repeat_seed}.csv", index=False)
        np.savez_compressed(run_dir / "features" / f"normalizer_seed_{repeat_seed}.npz", mean=mean, std=std)
        split_audit_rows.append(
            {
                "repeat_seed": int(repeat_seed),
                **{f"{split}_videos": int(meta[meta['split'].eq(split)]['video_id'].nunique()) for split in ['train', 'val', 'test']},
                **{f"{split}_danger_videos": int(meta[(meta['split'].eq(split)) & (meta['is_danger_clip'].eq(1))]['video_id'].nunique()) for split in ['train', 'val', 'test']},
            }
        )
        for objective, kind in args.model_specs:
            name = f"seed{repeat_seed}_{objective}_{kind}_hand_aug"
            y = multibin_y if objective == "multibin" else survival_y
            sample_weights = multibin_weights if objective == "multibin" else survival_weights
            print(f"training {name} on {device}")
            model, history, train_time_s, model_size_bytes, best = train_one(objective, kind, name, X, y, sample_weights, meta, run_dir, args, device)
            for row in history:
                row["repeat_seed"] = int(repeat_seed)
            all_history.extend(history)

            start = time.perf_counter()
            logits = []
            model.eval()
            with torch.no_grad():
                for idx in range(0, len(X), args.batch_size):
                    xb = torch.from_numpy(X[idx : idx + args.batch_size]).to(device)
                    logits.append(model(xb).detach().cpu())
            inference_s = time.perf_counter() - start
            logits = torch.cat(logits, dim=0).numpy()
            if objective == "multibin":
                probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
                pred = multibin_score_frame(meta, probs, name, train_time_s, inference_s)
            else:
                probs = 1.0 / (1.0 + np.exp(-logits))
                pred = survival_score_frame(meta, probs, name, train_time_s, inference_s)
            pred["repeat_seed"] = int(repeat_seed)
            pred.to_csv(run_dir / "features" / f"predictions_{name}.csv", index=False)
            score_cols = score_columns_for_objective(objective)
            all_metric_rows.extend(evaluate_prediction_frame(pred, score_cols, thresholds, persistence_values, repeat_seed))
            for row in ap_auc_rows(pred, score_cols):
                row["repeat_seed"] = int(repeat_seed)
                all_ap_rows.append(row)
            train_rows.append(
                {
                    "repeat_seed": int(repeat_seed),
                    "objective": objective,
                    "model_name": name,
                    "kind": kind,
                    "best_epoch": int(best["epoch"]),
                    "best_val_ap_future_0_15s": float(best["score"]),
                    "train_time_s": float(train_time_s),
                    "model_size_bytes": int(model_size_bytes),
                    "inference_ms_per_window": float(inference_s * 1000.0 / max(len(meta), 1)),
                }
            )

    metrics = pd.DataFrame(all_metric_rows)
    metrics = add_selection_score(metrics)
    metrics = with_base_model_name(metrics)
    ap_metrics = pd.DataFrame(all_ap_rows)
    history_df = pd.DataFrame(all_history)
    train_df = pd.DataFrame(train_rows)
    split_audit = pd.DataFrame(split_audit_rows)
    summary = summarize(
        metrics,
        ["objective", "base_model_name", "score_col", "threshold", "persistence_windows", "split"],
        ["selection_score", "pre_entry_recall", "early_03_recall", "early_05_recall", "event_precision", "false_alarms_per_min", "median_early_warning_s"],
    )

    metrics.to_csv(run_dir / "metrics" / "hand_detail_causal_metrics.csv", index=False)
    ap_metrics.to_csv(run_dir / "metrics" / "hand_detail_ap_metrics.csv", index=False)
    history_df.to_csv(run_dir / "metrics" / "hand_detail_training_history.csv", index=False)
    train_df.to_csv(run_dir / "metrics" / "hand_detail_training_summary.csv", index=False)
    split_audit.to_csv(run_dir / "metrics" / "hand_detail_split_audit.csv", index=False)
    summary.to_csv(run_dir / "metrics" / "hand_detail_summary.csv", index=False)
    write_json(run_dir / "metrics" / "hand_detail_feature_manifest.json", {"feature_columns": feature_cols, "hand_model": str(HAND_MODEL), "hand_columns": hand_cols})

    val_new = summary[summary["split"].eq("val")].sort_values("selection_score_mean", ascending=False)
    selected_val = val_new.iloc[0]
    selected_test = summary[
        summary["objective"].eq(selected_val["objective"])
        & summary["base_model_name"].eq(selected_val["base_model_name"])
        & summary["score_col"].eq(selected_val["score_col"])
        & summary["threshold"].eq(selected_val["threshold"])
        & summary["persistence_windows"].eq(selected_val["persistence_windows"])
        & summary["split"].eq("test")
    ].iloc[0].to_dict()
    best_test = summary[summary["split"].eq("test")].sort_values("selection_score_mean", ascending=False)
    write_repeated_summary(run_dir, selected_test, best_test, pd.DataFrame())

    if len(timing_df):
        hand_runtime = {
            "videos": int(len(timing_df)),
            "frames": int(timing_df["frames"].sum()),
            "seconds_total": float(timing_df["seconds"].sum()),
            "fps_mean_per_video": float(timing_df["fps"].mean()),
            "fps_overall": float(timing_df["frames"].sum() / max(timing_df["seconds"].sum(), 1e-6)),
        }
    else:
        hand_runtime = {
            "videos": int(base_meta["video_id"].nunique()),
            "frames": int(len(merged)),
            "seconds_total": None,
            "fps_mean_per_video": None,
            "fps_overall": None,
        }
    write_json(run_dir / "metrics" / "hand_runtime_summary.json", hand_runtime)
    print(run_dir)
    print(run_dir / "metrics" / "hand_runtime_summary.json")


## Fonction `parse_model_specs`

Cette cellule definit `parse_model_specs`. Elle prepare une partie du script.

In [ ]:
def parse_model_specs(values):
    specs = []
    for value in values:
        objective, kind = value.split(":", 1)
        specs.append((objective, kind))
    return specs


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Hand-detail repeated-split experiment using MediaPipe Hand Landmarker.")
    parser.add_argument("--pose-csv", default="runs/exp_070_physical_entry_baseline/features/pose_features.csv")
    parser.add_argument("--zones-json", default="annotations/zones.json")
    parser.add_argument("--sequence-index", default="runs/exp_072_physical_entry_seq30_focal/features/sequence_index.csv")
    parser.add_argument("--entry-times-csv", default="runs/exp_070_physical_entry_baseline/features/entry_times.csv")
    parser.add_argument("--old-score-run", default="runs/exp_088_physical_entry_final_score_full")
    parser.add_argument("--run-name", default="exp_103_hand_detail_repeated_split")
    parser.add_argument("--frame-features-csv", default="")
    parser.add_argument("--model-spec", action="append", default=["survival:gru", "survival:cnn1d", "multibin:tcn", "multibin:gru"])
    parser.add_argument("--frame-stride", type=int, default=3)
    parser.add_argument("--seq-len", type=int, default=10)
    parser.add_argument("--epochs", type=int, default=22)
    parser.add_argument("--patience", type=int, default=5)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.04)
    parser.add_argument("--focal-gamma", type=float, default=1.5)
    parser.add_argument("--threshold-min", type=float, default=0.05)
    parser.add_argument("--threshold-max", type=float, default=0.95)
    parser.add_argument("--threshold-step", type=float, default=0.05)
    parser.add_argument("--persistence-windows", nargs="+", type=int, default=[1, 2])
    parser.add_argument("--device", default="auto")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    args.model_specs = parse_model_specs(args.model_spec)
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_103_hand_detail_repeated_split_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["hand_detail_experiment.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
